In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/test_bench/checkpoints/pre_cell_3.pickle

In [ ]:
%%cudf.pandas.profile
### cell 3 ###

def combine_subset_of_data_from_multiple_years(question_of_interest, x_axis_title, include_2017=None):
    years = [2022, 2021, 2020, 2019, 2018]
    if include_2017 is not None:
        years.append(2017)
    responses_map = {
        2022: responses_df_2022,
        2021: responses_df_2021,
        2020: responses_df_2020,
        2019: responses_df_2019,
        2018: responses_df_2018,
        2017: responses_df_2017,
    }
    dfs = []
    for year in sorted(years):
        # compute percentages on GPU via cudf.pandas
        s = count_then_return_percent(responses_map[year], question_of_interest).sort_index()
        df_temp = s.reset_index(name='percentage').rename(columns={'index': x_axis_title})
        df_temp['year'] = year
        dfs.append(df_temp)
    df_combined = pd.concat(dfs, ignore_index=True)
    # ensure column order matches original
    df_combined = df_combined[['percentage', 'year', x_axis_title]]
    return df_combined

def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions(question_of_interest, include_2017=None):
    years = [2022, 2021, 2020, 2019, 2018]
    if include_2017 is not None:
        years.append(2017)
    responses_map = {
        2022: responses_df_2022,
        2021: responses_df_2021,
        2020: responses_df_2020,
        2019: responses_df_2019,
        2018: responses_df_2018,
        2017: responses_df_2017,
    }
    dfs = []
    for year in sorted(years):
        df_temp = grab_subset_of_data(responses_map[year], question_of_interest)
        df_temp['year'] = year
        dfs.append(df_temp)
    df_combined = pd.concat(dfs, ignore_index=True)
    df_combined_counts = df_combined.groupby('year').count().reset_index()
    return df_combined, df_combined_counts

def load_survey_data(base_dir, file_name, rows_to_skip=1):
    file_path = os.path.join(base_dir, file_name)
    df = pd.read_csv(file_path, low_memory=False, encoding='ISO-8859-1', skiprows=rows_to_skip)
    out_name = f"kaggle_survey_{base_dir[-5:-1]}.csv"
    if not glob.glob(out_name):
        df.to_csv(out_name, index=False)
    return df